# AURA — Evaluate Dreamed Trajectories

Load a trained checkpoint, run imagination scenarios, display GIFs,
and validate P1 success criteria.

In [ ]:
import os

if 'COLAB_GPU' in os.environ:
    !git clone https://github.com/SotoAlt/aura.git
    %cd aura
    !pip install -q -r world_model/requirements-colab.txt

    from google.colab import drive
    drive.mount('/content/drive')

    CHECKPOINT = '/content/drive/MyDrive/aura/checkpoints/aura-v0.1.ckpt'
    OUTPUT_DIR = '/content/drive/MyDrive/aura/eval_output'
else:
    CHECKPOINT = 'checkpoints/latest.ckpt'
    OUTPUT_DIR = 'eval_output'

In [ ]:
from world_model.eval import eval_report

summary = eval_report(CHECKPOINT, OUTPUT_DIR, n_frames=50)

## Display GIFs

In [ ]:
from pathlib import Path
from IPython.display import Image, display, HTML

output = Path(OUTPUT_DIR)
gifs = sorted(output.glob('*.gif'))

for gif in gifs:
    print(f'\n{gif.stem}:')
    display(Image(filename=str(gif)))

## Correlation Metrics

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open(output / 'metrics.json') as f:
    metrics = json.load(f)

# Plot correlation metrics per scenario
scenarios = list(metrics['scenarios'].keys())
brightness = [metrics['scenarios'][s]['mean_brightness'] for s in scenarios]
rms_corr = [metrics['scenarios'][s]['brightness_rms_corr'] for s in scenarios]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(scenarios, brightness)
ax1.set_ylabel('Mean Brightness')
ax1.set_title('Frame Brightness by Scenario')
ax1.tick_params(axis='x', rotation=30)

ax2.bar(scenarios, rms_corr)
ax2.set_ylabel('Correlation')
ax2.set_title('Brightness-RMS Correlation')
ax2.axhline(y=0.3, color='r', linestyle='--', label='P1 threshold')
ax2.legend()
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## P1 Success Criteria

In [ ]:
print('=' * 50)
print('P1 SUCCESS CRITERIA')
print('=' * 50)

criteria = metrics.get('p1_criteria', {})
all_pass = True
for k, v in criteria.items():
    status = 'PASS' if v else 'FAIL'
    if not v:
        all_pass = False
    print(f'  [{status}] {k}')

print()
cross = metrics.get('cross_scenario', {})
print(f'High vs Low brightness diff: {cross.get("high_vs_low_brightness_diff", 0):.4f}')
print(f'Any NaN: {cross.get("any_nan", False)}')
print(f'Any Inf: {cross.get("any_inf", False)}')

print()
if all_pass:
    print('ALL CRITERIA PASSED — P1 complete!')
else:
    print('SOME CRITERIA FAILED — needs more training or tuning')